# Biwenger Assistant

Interactive notebook to explore the market, your rivals' squads and your own, with points/market-value and points/release-clause ratios (current and previous season), performance rankings, fair market value, signing/selling suggestions, schedule/form, offers, and history.

All the loading/analysis logic lives in `biwenger_helpers.py` (shared with `02_generar_excel.py`) so as not to duplicate code; this notebook is mainly for exploring the data by hand.

**Before anything else**: copy `.env.example` to `.env` and fill in your credentials if you haven't already.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from dotenv import load_dotenv

import biwenger_helpers
from biwenger_client import BiwengerClient

load_dotenv()

EMAIL = os.getenv("BIWENGER_EMAIL")
PASSWORD = os.getenv("BIWENGER_PASSWORD")
LEAGUE_ID = os.getenv("BIWENGER_LEAGUE_ID") or None
OWN_TEAM_ID = os.getenv("BIWENGER_OWN_TEAM_ID") or None

pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Load data

The first time, this downloads everything from the API (market, every manager's squad, and the detail of each player involved) and saves it to `output/cache/*.json`. Subsequent times it reuses that cache instantly.

**It's resumable and stops itself at the rate limit**: with `parar_en_primer_limite=True` (enabled by default in this cell), as soon as the API cuts off due to too many requests (~200 in a row), the cell stops right there -- without sitting through long pauses -- and saves to cache whatever it managed to get so far (which is merged with what was fetched in previous runs). If `data` comes out incomplete, `build_dataframe` in the next cell simply shows fewer players than expected, it doesn't fail. Re-run this cell as many times as needed (at different moments if you like) to gradually fill in the cache until no one is missing.

**It refreshes itself every 24h**: if the cache is complete but older than `max_antiguedad_horas` (24 by default), this cell automatically refreshes it with genuinely fresh player data -- no need to remember to pass `forzar_refresco=True` by hand every day. Pass `max_antiguedad_horas=None` if you'd rather disable this and control the refresh yourself. An incomplete cache is always resumed, regardless of age.

And as always, `forzar_refresco=True` forces a full refresh right now, regardless of age.

In [ ]:
data = biwenger_helpers.load_league_data(
    EMAIL, PASSWORD, league_id=LEAGUE_ID, own_team_id=OWN_TEAM_ID,
    forzar_refresco=False, parar_en_primer_limite=True,
)

print(f"your league's score_id: {data['score_id']}")
print(f"your team_id: {data['mi_team_id']}")
print(f"your current balance: {data['balance']:,} €")

In [ ]:
df = biwenger_helpers.build_dataframe(data)
print(f"{len(df)} players in total ({df['Source'].nunique()} distinct origins)")
df.head(20)

In [ ]:
# Export the full DataFrame to Excel if you'd rather review it outside the notebook
# (for an already-formatted Excel with 3 sheets -- Mercado/Rivales/Mi equipo -- use 02_generar_excel.py)
# df.to_excel("output/mi_analisis.xlsx", index=False)

## 2. Explore

`df` has one row per player (market + every rival + your team), with the `Source` column to filter by. Main columns:

- `Points current season` / `Points previous season`, `Matches current season` / `Matches previous season`: totals for the current LaLiga season and the previous one (if they played in the first division). `Current season` / `Previous season` carry the real label (e.g. "Temporada 2025/2026") in case you need to check it -- the column name itself is always the same, it doesn't change per player. They're computed by MAJORITY across all players (`most_common_league_season`): if a player hasn't played this season or the previous one, they come out as NaN instead of carrying over points from 2+ seasons ago as if they were current.
- `Ratio pts/MV (current)` / `(previous)`: those total points divided by market value (in millions).
- `Ratio total pts/clause` / `Ratio avg pts/clause` (`current`/`previous`): two ways of looking at the same ratio against the release-clause price -- by TOTAL accumulated points, or by AVERAGE points (per match). Only for rivals'/your players, not market players. A player with few matches played (injury, recent signing) can look bad on the total but good on the average, and vice versa -- check both.
- `Potential score`: blends `Form` (70%) with points/match from the previous season (30%); this is the metric used by the rankings/recommenders further below.
- `Available` / `Status`: False if injured/suspended/discarded (see `ESTADOS_NO_DISPONIBLE` in `biwenger_helpers.py`; the recommenders exclude them by default). `'doubt'` doesn't count as unavailable -- it shows in `Status` but isn't filtered out, because it usually means they could still play.
- `Next opponent`, `Home/Away`, `Next match difficulty`: their next LaLiga match.
- `Price trend (7d) %`: change in their market value over the last 7 days.

**`Source` == `'Market'` means a genuinely FREE player** (no owner, directly buyable): market entries that DO have an owner (someone in your league put their player up for sale) don't count as 'Market' -- the only real way to sign them is via the release clause, so they only appear as `'Rival: <name>'` with their `Clause`. Also, `mercado/rivales` already exclude "junk": players without a top-division club, with value <= 400k, or inactive (haven't debuted this season and played less than half the matches last season). Your own team is never filtered -- there you see everything you have, active or not.

Some examples to get started:

In [ ]:
# Top 20 market bargains by points/market-value ratio
(
    df[df["Source"] == "Market"]
    .dropna(subset=["Ratio pts/MV (current)"])
    .sort_values("Ratio pts/MV (current)", ascending=False)
    .head(20)
)

In [ ]:
# Your squad sorted by Potential score (to see your weakest players)
df[df["Source"] == "My team"].sort_values("Potential score")

In [ ]:
# Best rival release clauses by total-pts/clause ratio (switch to "avg" to see by points per match)
(
    df[df["Source"].str.startswith("Rival:")]  # add "& (df["Position"] == "Forward")" to filter by position
    .dropna(subset=["Ratio total pts/clause (current)"])
    .sort_values("Ratio total pts/clause (current)", ascending=False)
    .head(20)
)

In [ ]:
# Look up a specific player by exact name
df[df["Player"] == "Nombre Apellido"]

## 3. Charts

In [ ]:
top20 = df.dropna(subset=["Ratio pts/MV (current)"]).nlargest(20, "Ratio pts/MV (current)")

fig, ax = plt.subplots(figsize=(8, 8))
ax.barh(top20["Player"], top20["Ratio pts/MV (current)"])
ax.invert_yaxis()
ax.set_xlabel("Ratio pts/MV (current)")
ax.set_title("Top 20 players by points / market-value ratio")
plt.tight_layout()
plt.show()

In [ ]:
posiciones = ["Goalkeeper", "Defender", "Midfielder", "Forward"]
datos = [df[df["Position"] == p]["Ratio pts/MV (current)"].dropna() for p in posiciones]

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(datos, tick_labels=posiciones)
ax.set_ylabel("Ratio pts/MV (current)")
ax.set_title("Distribution of the pts/VM ratio by position")
plt.tight_layout()
plt.show()

### Value vs performance -- bargains and overvalued players

Each point is a player: the X axis is market value, the Y axis is their `Potential score`. The dashed line is the overall trend (higher price, higher expected performance). Players well ABOVE the line perform better than their price suggests (bargains, in green); those well BELOW perform worse than expected for their price (overvalued, in red) -- the 8 outliers on each side are labeled with their name.

In [ ]:
plot_df = biwenger_helpers.unique_by_player(df.dropna(subset=["Market value", "Potential score"]))
plot_df = plot_df[plot_df["Market value"] > 0].copy()
plot_df["VM (M€)"] = plot_df["Market value"] / 1_000_000

COLOR_POSICION = {
    "Goalkeeper": "#1f77b4", "Defender": "#2ca02c",
    "Midfielder": "#ff7f0e", "Forward": "#d62728",
}

fig, ax = plt.subplots(figsize=(11, 7))
for pos, color in COLOR_POSICION.items():
    sub = plot_df[plot_df["Position"] == pos]
    ax.scatter(sub["VM (M€)"], sub["Potential score"], label=pos, alpha=0.45, color=color, s=22)

# overall trend (simple linear regression) to detect who deviates the most
x = plot_df["VM (M€)"].to_numpy()
y = plot_df["Potential score"].to_numpy()
pendiente, intercepto = np.polyfit(x, y, 1)
xs = np.linspace(x.min(), x.max(), 100)
ax.plot(xs, pendiente * xs + intercepto, "k--", alpha=0.5, linewidth=1, label="overall trend")

plot_df["_residuo"] = y - (pendiente * x + intercepto)
N_OUTLIERS = 8
chollos = plot_df.nlargest(N_OUTLIERS, "_residuo")
sobrevalorados = plot_df.nsmallest(N_OUTLIERS, "_residuo")

for _, row in chollos.iterrows():
    ax.annotate(row["Player"], (row["VM (M€)"], row["Potential score"]),
                fontsize=8, color="#1a7a1a", fontweight="bold",
                xytext=(5, 4), textcoords="offset points")
for _, row in sobrevalorados.iterrows():
    ax.annotate(row["Player"], (row["VM (M€)"], row["Potential score"]),
                fontsize=8, color="#b31212", fontweight="bold",
                xytext=(5, -9), textcoords="offset points")

ax.scatter(chollos["VM (M€)"], chollos["Potential score"], facecolors="none", edgecolors="#1a7a1a", s=90, linewidths=1.5)
ax.scatter(sobrevalorados["VM (M€)"], sobrevalorados["Potential score"], facecolors="none", edgecolors="#b31212", s=90, linewidths=1.5)

ax.set_xlabel("Market value (M€)")
ax.set_ylabel("Potential score")
ax.set_title("Value vs performance -- outliers labeled (green = bargain, red = overvalued)")
ax.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
plot_df.sort_values('_residuo', ascending=False).head(20)

In [ ]:
plot_df.sort_values('Ratio avg pts/clause (current)', ascending=False).head(20)

### Best and worst by position

The 5 with the best and the 5 with the worst `Potential score` in each position, across ALL players (market + rivals + your team -- the origin shows in parentheses). Useful for seeing at a glance who's standing out and who's struggling in each line of the team.

In [ ]:
N_POR_LADO = 5
posiciones = ["Goalkeeper", "Defender", "Midfielder", "Forward"]
df_unico = biwenger_helpers.unique_by_player(df)

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, pos in zip(axes.flat, posiciones):
    sub = df_unico[df_unico["Position"] == pos].dropna(subset=["Potential score"])
    peores = sub.nsmallest(N_POR_LADO, "Potential score").sort_values("Potential score", ascending=False)
    mejores = sub.nlargest(N_POR_LADO, "Potential score").sort_values("Potential score", ascending=True)
    combo = pd.concat([peores, mejores])
    etiquetas = combo["Player"] + " (" + combo["Source"].str.slice(0, 14) + ")"
    colores = ["#b31212"] * len(peores) + ["#1a7a1a"] * len(mejores)

    ax.barh(etiquetas, combo["Potential score"], color=colores)
    ax.set_title(pos)
    ax.axvline(0, color="grey", linewidth=0.8)
    ax.tick_params(axis="y", labelsize=8)

fig.suptitle("Best (green) and worst (red) by position -- Potential score", fontsize=13)
plt.tight_layout()
plt.show()

## 4. Signing suggestions

For each affordable, available candidate (a rival's release clause or a market price, within your current balance), this compares `Potential score` against your weakest player in that same position. Only candidates that would be an upgrade over that player show up. Rival release clauses that are currently locked (`Clause available now` == False, after a recent purchase/clause buyout -- the API would reject it) are always discarded, even if everything else fits.

Useful columns for deciding:
- `Balance after signing`: what your balance would be left after this signing.
- `Weak players in your position` / `Total in your position`: how many of your players in that position perform below your own median there -- more than one indicates a position with more than one weak link, not just the worst one.
- `Improvement per million spent`: the improvement normalized by cost. Use `ordenar_por="eficiencia"` to prioritize cheap signings that improve things gradually over one expensive signing that improves a lot at once.

This is an orientative suggestion, it doesn't take into account things like whether you already have room in your squad or the schedule of upcoming matches (for that check the `Next match difficulty` column of `df` in section 2).

In [ ]:
recomendaciones = biwenger_helpers.suggest_signings(df, data["balance"], top_n=15)
recomendaciones

In [ ]:
# Same calculation, but prioritizing efficient signings (improvement per million) instead of absolute improvement
biwenger_helpers.suggest_signings(df, data["balance"], top_n=15, ordenar_por="eficiencia")

## 5. Overall player ranking

`best_players` is a watchlist: who's performing best right now (by `Potential score`), regardless of price or whether you can afford it. Filter by position and/or by origin (`'Market'`, `'My team'`, or `'Rival: <name>'`). Excludes injured/suspended players by default.

In [ ]:
# Overall top 20, all positions and origins
biwenger_helpers.best_players(df, top_n=20)

In [ ]:
# Filtered example: best forwards available on the market
biwenger_helpers.best_players(df, posicion="Forward", origen="Market", top_n=15)

In [ ]:
# The best goalkeeper among all the ones we have data for (market + rivals + your team).
# If they're in 'Market' they're free (directly buyable); if 'Rival: X' you need the release clause.
mejor_portero = biwenger_helpers.best_players(df, posicion="Goalkeeper", top_n=1)
mejor_portero

In [ ]:
# Top 10 goalkeepers, to compare alternatives (not just the number 1)
biwenger_helpers.best_players(df, posicion="Goalkeeper", top_n=10)

## 6. Fair market value (bargains / overpriced)

For each player FOR SALE on the market, this estimates a "fair value" from how the rest of the players in your league (in the same position) relate performance (`Potential score`) to market value -- it uses the MEDIAN of that relationship as a reference. It compares that fair value with the asking price to detect bargains/overpriced players and suggest an offer.

It's a heuristic based on your own league, not Biwenger's "real" price -- treat it as guidance. `margen_pct` controls the % difference from which a player gets labeled Bargain/Expensive (20% by default).

In [ ]:
valor_justo = biwenger_helpers.estimate_fair_value(df, margen_pct=20)
valor_justo

In [ ]:
# Only the bargains, sorted from best to worst
valor_justo[valor_justo["Rating"] == "Bargain"]

### How much to offer (market only -- release clauses are a fixed amount, you don't bid)

`suggest_market_offer` combines three things for each free player:
- Your **estimated fair value** (same as above): it never suggests bidding above this.
- **Real competition**: how many rivals have a `maximumBid` (standings bid limit, can be higher than their current balance) above the asking price. If there's competition, it suggests a margin over the asking price so you don't lose the player by a small amount.
- **Real comparables**: the median of what the current owners of similar players (same position, similar value) actually paid for them.

It's for guidance only -- we don't know who is REALLY interested in each player, only who could afford it.

In [ ]:
biwenger_helpers.suggest_market_offer(df, data)

## 7. Selling suggestions

The counterpart to section 6, but for YOUR squad: compares each of your players' OFFICIAL market value against what their current performance would justify (same per-position reference median). If it's well above, the player is overvalued -- selling now takes advantage of a price that probably won't hold if their performance doesn't improve.

It also cross-references the **real offers** you've received (section 8): if someone is already offering you the estimated fair value or more, selling is recommended even if the overvaluation % alone wouldn't reach the threshold -- a real offer in hand outweighs the heuristic.

In [ ]:
ventas = biwenger_helpers.suggest_sales(df, data=data, margen_pct=20)
ventas

In [ ]:
# Only the ones worth selling right now
ventas[ventas["Recommendation"] == "Sell now"]

## 8. Offers

Active purchase offers on the market: `'recibidas'` are the ones other managers have made you for your players, `'enviadas'` are the ones you've made. This library doesn't include accepting/rejecting -- do that from the app; this is just to list them without having to open it.

Note: Biwenger doesn't always identify the bidder in this response, so `From`/`To` may come out as `'Unknown'`.

In [ ]:
biwenger_helpers.offers(data, tipo="recibidas")

In [ ]:
biwenger_helpers.offers(data, tipo="enviadas")

## 9. History

Unlike the cache from section 1 (which gets overwritten on every refresh), `save_snapshot` saves a dated copy to `output/history/<timestamp>/`, so you can compare how your team/the market evolves over time. Use it right after a `forzar_refresco=True`.

In [ ]:
# Uncomment to save a dated snapshot of the current data
# biwenger_helpers.save_snapshot(data)

biwenger_helpers.list_snapshots()

In [ ]:
# Example: compare your current squad against a previous snapshot.
# Replace the name with a real one from list_snapshots().
# previous_df = biwenger_helpers.build_dataframe(biwenger_helpers.load_snapshot("20260828_120000"))
# comparison = df.merge(
#     previous_df[["player_id", "Potential score", "Market value"]],
#     on="player_id", suffixes=("", " (before)"),
# )
# comparison["Change in potential"] = comparison["Potential score"] - comparison["Potential score (before)"]
# comparison[comparison["Source"] == "My team"].sort_values("Change in potential")

## 10. Operations (bid / buy out clause) -- REAL actions on your league

`place_offer` runs in **dry-run mode by default**: without `confirm=True` nothing is sent, it only prints the payload that would be sent. Always review the dry-run before confirming -- this actually spends/commits your balance and **cannot be undone** from here.

It needs its own login (it doesn't reuse the session from `load_league_data`, which is read-only).

In [ ]:
cliente_operaciones = BiwengerClient(EMAIL, PASSWORD, league_id=LEAGUE_ID)
biwenger_helpers.login_and_resolve_league(cliente_operaciones, LEAGUE_ID)
print("Session ready. Balance (already loaded in step 1):", data["balance"])

In [ ]:
# Example: bid on / buy the clause for a player from 'recomendaciones', 'valor_justo' or whichever you like.
# 'Seller (id)' from those tables is what goes in 'to'.
player_id = None  # <- put the player_id here
amount = None  # <- your offer amount (or the release-clause price; in valor_justo, 'Suggested offer')
seller_id = None  # <- 'Seller (id)' from the row you're interested in

if player_id and amount and seller_id:
    cliente_operaciones.place_offer(player_id, amount, to=seller_id)
    # Once the payload above looks correct to you, uncomment this line
    # to actually execute it (spends/commits your balance):
    # cliente_operaciones.place_offer(player_id, amount, to=seller_id, confirm=True)
else:
    print("Fill in player_id / amount / seller_id before running this cell.")